## Load the tokenizer

In [3]:
from transformers import AutoTokenizer
tok = AutoTokenizer.from_pretrained("HuggingFaceTB/SmolLM2-135M")

/Users/ibrahim/paper-companion/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### See how tokens are generated

In [4]:
tokens = tok.convert_ids_to_tokens(tok("Ibraheem uses tokenization.")["input_ids"])
print(len(tokens), tokens)

8 ['I', 'bra', 'he', 'em', 'Ġuses', 'Ġtoken', 'ization', '.']


## Load the model

In [5]:
from transformers import AutoModelForCausalLM
model = AutoModelForCausalLM.from_pretrained("HuggingFaceTB/SmolLM2-135M")

W1111 16:41:46.483000 50414 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.
Loading weights: 100%|██████████| 272/272 [00:00<00:00, 5688.10it/s]


In [9]:
import torch
inputs = tok("The capital of France is", return_tensors="pt")
with torch.no_grad():
    logits = model(input_ids=inputs["input_ids"], attention_mask=inputs["attention_mask"]).logits
print(logits.shape)

torch.Size([1, 5, 49152])


### Score of the next word

In [8]:
top = torch.topk(logits[0, -1], k=5)
for score, i in zip(top.values, top.indices):
    print(f"{score.item():7.2f}  {tok.decode(i)!r}")

  14.12  ' the'
  12.94  ' Paris'
  12.88  ' located'
  12.38  ' called'
  12.31  ' a'


### Softmax of the next word

In [10]:
probs = torch.softmax(logits[0, -1], dim=-1)
top = torch.topk(probs, k=5)
for p, i in zip(top.values, top.indices):
    print(f"{p.item():6.1%}  {tok.decode(i)!r}")

 26.2%  ' the'
  8.0%  ' Paris'
  7.5%  ' located'
  4.6%  ' called'
  4.3%  ' a'


### Repetitive loop sentence construction

In [17]:
ids = inputs["input_ids"]
for _ in range(25):
    with torch.no_grad():
        next_logits = model(input_ids=ids).logits[0, -1]
    next_id = next_logits.argmax()
    ids = torch.cat([ids, next_id.view(1, 1)], dim=1)
print(tok.decode(ids[0]))

The capital of France is the city of Paris. It is the largest city in France and the capital of the department of department of department of department of


In [22]:
ids = inputs["input_ids"]
for _ in range(25):
    with torch.no_grad():
        next_logits = model(input_ids=ids).logits[0, -1]
    probs = torch.softmax(next_logits / 0.6, dim=-1)
    next_id = torch.multinomial(probs, num_samples=1)[0]
    ids = torch.cat([ids, next_id.view(1, 1)], dim=1)
print(tok.decode(ids[0]))

The capital of France is the largest city in the country, 7,000,000 inhabitants and the largest in the European Union


### HuggingFace model loop shortcut

In [24]:
out = model.generate(**inputs, max_new_tokens=25, do_sample=False, use_cache=False)
print(tok.decode(out[0]))

The capital of France is the city of Paris. It is the largest city in France and the capital of the department of department of department of department of
